# Imagini și sunete cu modele de difuzie

Laborator practic în care generăm imagini și sunete cu modele de difuzie cu ponderi deschise, în Python, cu bibliotecile Diffusers și Transformers, pe propriul calculator sau în Google Colab.

Notebookul conține, în ordine, codul tuturor paginilor laboratorului [Imagini și sunete cu modele de difuzie](https://inteligentartificiala.ro/laboratoare/difuzie-imagini-si-sunete/prezentare/), de pe [InteligentArtificiala.ro](https://inteligentartificiala.ro). Explicațiile fiecărui pas sunt pe pagini.

În Google Colab, codul rulează mult mai repede pe o placă grafică. Dacă nu este deja aleasă, se alege din meniul **Runtime**, opțiunea **Change runtime type**, o variantă cu GPU. Celulele se rulează în ordine, de sus în jos.

In [ ]:
%pip install -q -U diffusers transformers accelerate

## [Introducere](https://inteligentartificiala.ro/laboratoare/difuzie-imagini-si-sunete/prezentare/)

### Prima imagine

Cel mai scurt drum de la nimic la o imagine este un pipeline.

In [ ]:
import torch
from diffusers import DiffusionPipeline

# Placa grafică NVIDIA (cuda), placa grafică a unui cip Apple Silicon (mps) sau, dacă nu există niciuna, procesorul.
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32

# La prima rulare, fișierele modelului (aproximativ 2,7 GB) se descarcă de pe Hugging Face Hub.
pipe = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", dtype=dtype, variant="fp16")
pipe = pipe.to(device)

prompt = "a watercolor painting of an orange cat sleeping on a park bench"
generator = torch.Generator("cpu").manual_seed(0)
image = pipe(prompt, generator=generator).images[0]
image.save("prima_imagine.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("prima_imagine.png"))

## [Procesul de difuzie, pas cu pas](https://inteligentartificiala.ro/laboratoare/difuzie-imagini-si-sunete/procesul-de-difuzie/)

In [ ]:
# Modelele paginilor anterioare nu mai sunt necesare, așa că le scoatem din memorie.
%reset -f
import gc

import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Programul de zgomot

Diffusers separă rețeaua de **metoda de eșantionare** (_scheduler_). Rețeaua, aici un `UNet2DModel`, primește o imagine cu zgomot și pasul $t$ și estimează zgomotul. Metoda de eșantionare păstrează programul de zgomot și calculează, din estimarea rețelei, imaginea de la pasul următor. Codul de mai jos încarcă doar metoda de eșantionare și afișează, pentru câțiva pași, coeficienții din formula procesului direct, $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\, \varepsilon$.

In [ ]:
from diffusers import DDPMScheduler

# Metoda de eșantionare DDPM, cu programul de zgomot folosit la antrenarea modelului.
scheduler = DDPMScheduler.from_pretrained("google/ddpm-cat-256")
print(f"beta_schedule = {scheduler.config.beta_schedule}, T = {scheduler.config.num_train_timesteps}")

for t in [0, 250, 500, 750, 999]:
    alpha_bar = scheduler.alphas_cumprod[t]  # ᾱ_t
    print(f"t = {t:3d}: {alpha_bar.sqrt():.3f} × imaginea + {(1 - alpha_bar).sqrt():.3f} × zgomotul")

### Procesul invers, scris de mână

Codul de mai jos pornește de la zgomot pur, $x_T$, și face 50 de pași cu metoda **DDIM**, a cărei regulă este descrisă pe pagina despre [generare](https://inteligentartificiala.ro/modele-de-difuzie/ce-sunt-modelele-de-difuzie/#generarea-de-la-zgomot-la-imagine). La câțiva pași, codul salvează două imagini într-o grilă: imaginea cu zgomot, $x_t$, și estimarea imaginii curate, $\hat x_0$, calculată de metoda de eșantionare din zgomotul estimat de rețea.

In [ ]:
import torch
from diffusers import DDIMScheduler, UNet2DModel
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model_id = "google/ddpm-cat-256"
unet = UNet2DModel.from_pretrained(model_id).to(device)
scheduler = DDIMScheduler.from_pretrained(model_id)  # aceeași rețea, cu metoda de eșantionare DDIM
scheduler.set_timesteps(50)


def to_image(x):
    # Un tensor cu valori între -1 și 1 devine o imagine cu pixeli între 0 și 255.
    x = (x[0].clamp(-1, 1) + 1) / 2
    return Image.fromarray((x.permute(1, 2, 0).cpu().numpy() * 255).round().astype("uint8"))


generator = torch.Generator("cpu").manual_seed(0)
x = torch.randn((1, 3, 256, 256), generator=generator).to(device)  # x_T, zgomot pur
shown = [0, 10, 20, 30, 40, 49]  # pașii afișați în grilă
grid = Image.new("RGB", (len(shown) * 256, 2 * 256))
for i, t in enumerate(scheduler.timesteps):
    with torch.no_grad():
        noise_pred = unet(x, t).sample  # zgomotul estimat de rețea, ε̂
    step = scheduler.step(noise_pred, t, x)
    if i in shown:
        grid.paste(to_image(x), (256 * shown.index(i), 0))  # sus: x_t
        grid.paste(to_image(step.pred_original_sample), (256 * shown.index(i), 256))  # jos: x̂_0
    x = step.prev_sample  # x_{t-1}, cu ceva mai puțin zgomot

grid.save("pasi.png")
to_image(x).save("pisica.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("pasi.png"))
ipd.display(ipd.Image("pisica.png"))

### DDPM și DDIM

Aceeași rețea poate fi folosită cu metode de eșantionare diferite, fără nicio antrenare suplimentară. Codul de mai jos generează câte o pisică prin DDPM, cu toți cei 1.000 de pași, și prin DDIM, cu 50 de pași, pornind de la același zgomot, și măsoară cât durează fiecare. Pentru că pipeline-urile `DDPMPipeline` și `DDIMPipeline` fac aceiași pași ca mai sus, codul este mai scurt.

In [ ]:
import time

import torch
from diffusers import DDIMPipeline, DDIMScheduler, DDPMPipeline
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ddpm = DDPMPipeline.from_pretrained("google/ddpm-cat-256").to(device)
# Aceeași rețea, cu metoda de eșantionare DDIM.
ddim = DDIMPipeline(unet=ddpm.unet, scheduler=DDIMScheduler.from_config(ddpm.scheduler.config))

grid = Image.new("RGB", (2 * 256, 256))
for column, (name, pipe, steps) in enumerate([("DDPM", ddpm, 1000), ("DDIM", ddim, 50)]):
    start = time.time()
    generator = torch.Generator("cpu").manual_seed(0)
    image = pipe(num_inference_steps=steps, generator=generator).images[0]
    print(f"{name}: {steps} de pași în {time.time() - start:.1f} secunde")
    grid.paste(image, (256 * column, 0))
grid.save("ddpm_ddim.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("ddpm_ddim.png"))

### Procesul direct

Procesul direct nu are nevoie de rețea: este doar formula programului de zgomot. Metoda `add_noise` o aplică direct pentru orice pas $t$, fără pașii de dinainte, exact ca la antrenare. Codul de mai jos adaugă zgomot pisicii generate mai sus, salvată în `pisica.png`, și compară rezultatul cu formula scrisă de mână.

In [ ]:
import numpy as np
import torch
from diffusers import DDPMScheduler
from PIL import Image

scheduler = DDPMScheduler.from_pretrained("google/ddpm-cat-256")
# Pisica generată mai sus, cu pixelii aduși între -1 și 1.
x0 = torch.from_numpy(np.array(Image.open("pisica.png"))).permute(2, 0, 1)[None].float() / 127.5 - 1
noise = torch.randn(x0.shape, generator=torch.Generator().manual_seed(0))  # ε

grid = Image.new("RGB", (5 * 256, 256))
for column, t in enumerate([0, 250, 500, 750, 999]):
    x_t = scheduler.add_noise(x0, noise, torch.tensor([t]))
    alpha_bar = scheduler.alphas_cumprod[t]
    by_hand = alpha_bar.sqrt() * x0 + (1 - alpha_bar).sqrt() * noise  # x_t = √ᾱ_t · x_0 + √(1 − ᾱ_t) · ε
    print(f"t = {t:3d}: diferența maximă față de formula scrisă de mână este {(x_t - by_hand).abs().max():.1e}")
    pixels = ((x_t[0].clamp(-1, 1) + 1) * 127.5).permute(1, 2, 0).round().byte().numpy()
    grid.paste(Image.fromarray(pixels), (256 * column, 0))
grid.save("procesul_direct.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("procesul_direct.png"))

## [De la text la imagine](https://inteligentartificiala.ro/laboratoare/difuzie-imagini-si-sunete/imagini-din-text/)

In [ ]:
# Modelele paginilor anterioare nu mai sunt necesare, așa că le scoatem din memorie.
%reset -f
import gc

import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Textul devine vectori

Repozitoriul unui model din Diffusers are câte un director pentru fiecare componentă: `tokenizer`, `text_encoder`, `unet`, `vae` și `scheduler`. Fiecare se poate încărca separat, cu argumentul `subfolder`. Codul de mai jos transformă descrierea în vectori cu tokenizatorul și cu codificatorul de text al lui [CLIP](https://inteligentartificiala.ro/modele-de-difuzie/generarea-imaginilor/#cum-ajunge-textul-în-imagine), care vin din biblioteca Transformers.

In [ ]:
import torch
from transformers import CLIPTextModel, CLIPTokenizer

model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", variant="fp16", dtype=torch.float32)

prompt = "a watercolor painting of an orange cat sleeping on a park bench"
tokens = tokenizer(
    prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt"
)
print("Primele tokenuri:", tokenizer.convert_ids_to_tokens(tokens.input_ids[0, :15]))
with torch.no_grad():
    embeddings = text_encoder(tokens.input_ids).last_hidden_state
print("Vectorii textului:", tuple(embeddings.shape))

### Generarea, cu ghidarea fără clasificator

In [ ]:
import torch
from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel
from PIL import Image
from transformers import CLIPTextModel, CLIPTokenizer

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", variant="fp16", dtype=dtype).to(device)
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", variant="fp16", dtype=dtype).to(device)
vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", variant="fp16", dtype=dtype).to(device)
scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")


def encode(text):
    ids = tokenizer(
        text, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt"
    ).input_ids
    with torch.no_grad():
        return text_encoder(ids.to(device)).last_hidden_state


cond = encode("a watercolor painting of an orange cat sleeping on a park bench")  # c
uncond = encode("")  # ∅, textul gol
w = 7.5  # scala de ghidare

generator = torch.Generator("cpu").manual_seed(0)
latents = torch.randn((1, 4, 64, 64), generator=generator).to(device, dtype)  # zgomot pur, în spațiul latent
scheduler.set_timesteps(50)
for t in scheduler.timesteps:
    with torch.no_grad():
        noise_cond = unet(latents, t, encoder_hidden_states=cond).sample
        noise_uncond = unet(latents, t, encoder_hidden_states=uncond).sample
    noise_pred = noise_uncond + w * (noise_cond - noise_uncond)
    latents = scheduler.step(noise_pred, t, latents).prev_sample

with torch.no_grad():
    image = vae.decode(latents / vae.config.scaling_factor).sample  # din spațiul latent înapoi în pixeli
pixels = ((image[0].clamp(-1, 1) + 1) * 127.5).permute(1, 2, 0).round().byte().cpu().numpy()
Image.fromarray(pixels).save("pisica_de_mana.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("pisica_de_mana.png"))

### Scala de ghidare

Parametrul `guidance_scale` al pipeline-ului este scala $w$ din formula de mai sus, cu valoarea implicită 7,5. Codul de mai jos generează aceeași imagine, cu același zgomot de pornire, pentru patru valori ale scalei și le pune una lângă alta.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, dtype=dtype, variant="fp16").to(device)

prompt = "a watercolor painting of an orange cat sleeping on a park bench"
scales = [1.0, 3.0, 7.5, 15.0]
grid = Image.new("RGB", (len(scales) * 512, 512))
for column, w in enumerate(scales):
    generator = torch.Generator("cpu").manual_seed(0)
    image = pipe(prompt, guidance_scale=w, generator=generator).images[0]
    grid.paste(image, (512 * column, 0))
grid.save("scala_de_ghidare.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("scala_de_ghidare.png"))

### De la imagine la imagine

În generarea **de la imagine la imagine** (_image-to-image_), procesul invers nu pornește de la zgomot pur, ci de la o imagine existentă, trecută prin procesul direct până la un pas intermediar. Parametrul **`strength`** (intensitate) alege acest pas. Cu 50 de pași și o intensitate de 0,3, pipeline-ul adaugă zgomot cât pentru ultimii 15 pași și rulează doar acești 15 pași. Codul de mai jos pornește de la pisica generată mai sus, cu o descriere nouă, pentru trei valori ale intensității.

In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(model_id, dtype=dtype, variant="fp16").to(device)

init_image = Image.open("pisica_de_mana.png").convert("RGB")  # imaginea generată mai sus, fără pipeline
prompt = "an oil painting of an orange cat sleeping on a park bench, thick brush strokes"
strengths = [0.3, 0.6, 0.9]
grid = Image.new("RGB", ((len(strengths) + 1) * 512, 512))
grid.paste(init_image, (0, 0))
for column, strength in enumerate(strengths, start=1):
    generator = torch.Generator("cpu").manual_seed(0)
    image = pipe(prompt, image=init_image, strength=strength, generator=generator).images[0]
    grid.paste(image, (512 * column, 0))
grid.save("de_la_imagine_la_imagine.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("de_la_imagine_la_imagine.png"))

## [Sunete din text](https://inteligentartificiala.ro/laboratoare/difuzie-imagini-si-sunete/sunete-din-text/)

In [ ]:
# Modelele paginilor anterioare nu mai sunt necesare, așa că le scoatem din memorie.
%reset -f
import gc

import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Primul sunet

Codul de mai jos generează 10 secunde de sunet pornind de la o descriere și le salvează într-un fișier WAV.

In [ ]:
import scipy.io.wavfile
import torch
from diffusers import AudioLDM2Pipeline

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32

# La prima rulare, fișierele modelului (aproximativ 4,5 GB) se descarcă de pe Hugging Face Hub.
pipe = AudioLDM2Pipeline.from_pretrained("cvssp/audioldm2", dtype=dtype).to(device)

prompt = "a dog barking in the distance while rain falls on a tin roof"
generator = torch.Generator("cpu").manual_seed(0)
audio = pipe(prompt, negative_prompt="low quality", audio_length_in_s=10.0, generator=generator).audios[0]

rate = pipe.vocoder.config.sampling_rate  # eșantioane pe secundă
print(f"{len(audio)} de eșantioane, adică {len(audio) / rate:.1f} secunde la {rate} Hz")
scipy.io.wavfile.write("sunet.wav", rate=rate, data=audio)

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Audio("sunet.wav"))

### De la reprezentarea latentă la sunet

Cu `output_type="latent"`, pipeline-ul se oprește după procesul invers și întoarce reprezentarea latentă curată, fără să o decodeze. Codul de mai jos face de mână ultimii doi pași, decodarea cu VAE-ul și vocoderul, și desenează spectrograma mel dintre ei.

In [ ]:
import matplotlib.pyplot as plt
import scipy.io.wavfile
import torch
from diffusers import AudioLDM2Pipeline

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
pipe = AudioLDM2Pipeline.from_pretrained("cvssp/audioldm2", dtype=dtype).to(device)

prompt = "a slow piano melody with soft rain in the background"
generator = torch.Generator("cpu").manual_seed(0)
latents = pipe(
    prompt, negative_prompt="low quality", audio_length_in_s=10.0, generator=generator, output_type="latent"
).audios
with torch.no_grad():
    mel = pipe.vae.decode(latents / pipe.vae.config.scaling_factor).sample  # spectrograma mel
    audio = pipe.mel_spectrogram_to_waveform(mel)[0].numpy()  # vocoderul HiFi-GAN
print("Reprezentarea latentă:", tuple(latents.shape))
print("Spectrograma mel:     ", tuple(mel.shape))
print("Forma de undă:        ", audio.shape)

plt.figure(figsize=(10, 3))
plt.imshow(mel[0, 0].T.float().cpu().numpy(), origin="lower", aspect="auto", cmap="magma")
plt.xlabel("cadre de câte 10 ms")
plt.ylabel("benzi mel")
plt.savefig("spectrograma.png", bbox_inches="tight")
scipy.io.wavfile.write("pian.wav", rate=pipe.vocoder.config.sampling_rate, data=audio)

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Audio("pian.wav"))